# Auditoría reproducible de la cohorte experimental v1
Comprueba selección, recuperación, integridad y corte temporal. No ejecuta modelos ni produce resultados científicos.

In [ ]:
import csv, json, hashlib
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
COHORT = ROOT / 'data/processed/experimental_cohort_v1'
SAFE = COHORT / 'temporal_safe'

In [ ]:
manifest = json.loads((COHORT / 'cohort_manifest.json').read_text())
quality = json.loads((SAFE / 'temporal_cohort_report.json').read_text())
manifest, quality

In [ ]:
assert manifest['selected_profiles'] == 50
assert manifest['candidate_opportunities'] == 1000
assert quality['initial_releases'] == 1317
assert quality['labels_missing_candidate'] == 0
assert quality['profile_awards_after_cutoff'] == 0
assert quality['test_candidates_before_cutoff'] == 0
assert quality['empty_candidate_text'] == 0
assert quality['empty_profile_text'] == 0
print('Controles principales: OK')

In [ ]:
with (COHORT / 'initial_release_results.csv').open() as f:
    releases = list(csv.DictReader(f))
hash_errors = sum(hashlib.sha256(Path(r['cache_path']).read_bytes()).hexdigest() != r['sha256'] for r in releases)
{'releases': len(releases), 'failed': sum(r['status']=='failed' for r in releases), 'hash_errors': hash_errors}

## Interpretación
La cohorte es apta para ejecutar los baselines. Las 28 discrepancias de fecha compilada/inicial se resuelven usando la fecha inicial. Las etiquetas siguen siendo positivas débiles y no convierten el resto del pool en negativos verdaderos.